**ENTRENAMIENTO DEL DETECTOR DE MATRÍCULAS CON RT-DETRv2**

En este cuaderno se describe cómo se entrenó el detector de matrículas con RT-DETRv2 (`best.pth`) que se utiliza después en el cuaderno de lectura y seguimiento en vídeo.

El flujo de trabajo se estructura en:
1. **Configuración inicial**: Rutas, pesos preentrenados, dispositivo (GPU/CPU) y directorio de resultados.
2. **Entrenamiento**: *Fine-tuning* de `rtdetrv2_r18vd` (preentrenado en COCO) sobre el conjunto de matrículas, con una única clase (`plate`).
3. **Evaluación**: Curvas de AP/AP50/AP75 y pérdida por época, junto con gráficas adicionales de progreso y las métricas COCO finales.
4. **Validación del mejor modelo**: Evaluación de `best.pth` sobre el conjunto de validación.
5. **Prueba de inferencia**: Lote de imágenes de validación con las cajas reales y las predichas.

Todas las gráficas y el lote de imágenes se guardan en el directorio `RT-DETR_resultado_entrenamiento`.

**CONFIGURACIÓN INICIAL: RUTAS Y PESOS PREENTRENADOS**

Se definen las rutas del fichero de configuración, de la carpeta de salida del entrenamiento y de los pesos de partida. Si los pesos no existen, se descargan de GitHub (`rtdetrv2_r18vd_120e_coco_rerun_48.1.pth`).

Además se crea (si no existe) el directorio `RT-DETR_resultado_entrenamiento`, donde se guardarán las gráficas y las imágenes de resultados.

In [1]:
import os
import sys
import subprocess
import urllib.request
import torch
from pathlib import Path

# =====================================================
# DIRECTORIO DE RESULTADOS (se crea si no existe)
# =====================================================

RESULTS_DIR = Path("RT-DETR_resultado_entrenamiento").resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Directorio de resultados:", RESULTS_DIR)

# =====================================================
# PESOS PREENTRENADOS
# =====================================================


PRETRAINED_URL = (
    "https://github.com/lyuwenyu/storage/releases/download/"
    "v0.2/rtdetrv2_r18vd_120e_coco_rerun_48.1.pth"
)

CONFIG = Path("RT-DETR/rtdetrv2_pytorch/configs/rtdetrv2/rtdetrv2_r18vd_120e_plates.yml").resolve()
OUTPUT_DIR = Path("RT-DETR/rtdetrv2_pytorch/output/rtdetrv2_r18vd_120e_plate").resolve()
PRETRAINED = Path("RT-DETR/rtdetrv2_pytorch/output/rtdetrv2_r18vd_120e_plate/best.pth").resolve()
RTDETR_PATH = os.path.abspath("RT-DETR/rtdetrv2_pytorch")

if not PRETRAINED.exists():

    print("Descargando pesos preentrenados...")

    urllib.request.urlretrieve(
        PRETRAINED_URL,
        PRETRAINED
    )

print("Pesos descargados:", PRETRAINED)


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Dispositivo:", DEVICE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Directorio de resultados: C:\Users\vidal\OneDrive\Escritorio\UNI\4ano\VC\Practicas_VC\P4\RT-DETR_resultado_entrenamiento
Pesos descargados: C:\Users\vidal\OneDrive\Escritorio\UNI\4ano\VC\Practicas_VC\P4\RT-DETR\rtdetrv2_pytorch\output\rtdetrv2_r18vd_120e_plate\best.pth
Dispositivo: cuda
GPU: NVIDIA GeForce RTX 3070


**ENTRENAMIENTO DEL MODELO**

La función `run_training` lanza `tools/train.py` del repositorio RT-DETR como subproceso, usando la configuración `rtdetrv2_r18vd_120e_plates.yml`. Los aspectos principales son:
- **`--use-amp`**: Entrenamiento con precisión mixta para reducir el uso de memoria y acelerar el proceso.
- **`--seed=0`**: Semilla fija para que el entrenamiento sea reproducible.
- **`-t`**: Pesos de partida para el *fine-tuning* (checkpoint preentrenado en COCO).
- **`-r`**: Permite reanudar un entrenamiento interrumpido desde `last.pth`.

La salida del entrenamiento se muestra en directo en el cuaderno.

In [2]:
# =====================================================
# ENTRENAMIENTO
# =====================================================

def run_training(resume=None):

    cmd = [
        sys.executable,
        "tools/train.py",
        "-c", str(CONFIG),
        "--use-amp",
        "--seed=0"
    ]

    if resume is not None:
        cmd += ["-r", str(resume)]

    else:

        cmd += ["-t", str(PRETRAINED)]

    env = os.environ.copy()

    env["CUDA_VISIBLE_DEVICES"] = "0"
    env["PYTHONUNBUFFERED"] = "1"

    print("\nIniciando entrenamiento RT-DETRv2...")
    print("Configuración:", CONFIG)
    print("Pesos iniciales:", PRETRAINED)

    process = subprocess.Popen(
        cmd,
        cwd=str(RTDETR_PATH),
        env=env,
        text=True,
        bufsize=1,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )

    for line in process.stdout:
        print(line, end="")

    process.wait()

    if process.returncode != 0:
        raise RuntimeError(
            f"Error durante el entrenamiento: {process.returncode}"
        )

    print("\nEntrenamiento finalizado correctamente.")

**EJECUCIÓN DEL ENTRENAMIENTO**

Se lanza el entrenamiento. La primera línea (comentada) inicia un entrenamiento desde los pesos preentrenados; la segunda reanuda el entrenamiento desde el último checkpoint (`last.pth`).

In [3]:
# =====================================================
# EJECUTAR
# =====================================================

run_training()

# Para reanudar un entrenamiento interrumpido:
#run_training(resume=OUTPUT_DIR / "last.pth")


Iniciando entrenamiento RT-DETRv2...
Configuración: C:\Users\vidal\OneDrive\Escritorio\UNI\4ano\VC\Practicas_VC\P4\RT-DETR\rtdetrv2_pytorch\configs\rtdetrv2\rtdetrv2_r18vd_120e_plates.yml
Pesos iniciales: C:\Users\vidal\OneDrive\Escritorio\UNI\4ano\VC\Practicas_VC\P4\RT-DETR\rtdetrv2_pytorch\output\rtdetrv2_r18vd_120e_plate\best.pth
Not init distributed mode.
cfg:  {'task': 'detection', '_model': None, '_postprocessor': None, '_criterion': None, '_optimizer': None, '_lr_scheduler': None, '_lr_warmup_scheduler': None, '_train_dataloader': None, '_val_dataloader': None, '_ema': None, '_scaler': None, '_train_dataset': None, '_val_dataset': None, '_collate_fn': None, '_evaluator': None, '_writer': None, 'num_workers': 0, 'batch_size': None, '_train_batch_size': None, '_val_batch_size': None, '_train_shuffle': None, '_val_shuffle': None, 'resume': None, 'tuning': 'C:\\Users\\vidal\\OneDrive\\Escritorio\\UNI\\4ano\\VC\\Practicas_VC\\P4\\RT-DETR\\rtdetrv2_pytorch\\output\\rtdetrv2_r18vd_120

**EVALUACIÓN DEL MODELO ENTRENADO**

Se leen las métricas de evaluación COCO por época desde `log.txt`. Si el entrenamiento se reanudó, una misma época puede aparecer repetida, por lo que se conserva la última aparición de cada una. Se obtienen:
- **AP (0.50:0.95)**: Precisión media promediada entre IoU 0.5 y 0.95.
- **AP50 / AP75**: Precisión media con IoU de 0.5 y 0.75, respectivamente.
- **AR**: Recall medio para distintos números máximos de detecciones y tamaños de objeto.

Se muestra la gráfica principal de AP por época junto con la pérdida de entrenamiento y se guarda en `RT-DETR_resultado_entrenamiento`.

In [ ]:
import json
import matplotlib.pyplot as plt

LOG_FILE = OUTPUT_DIR / "log.txt"

rows = [
    json.loads(line)
    for line in open(LOG_FILE, encoding="utf-8")
    if line.strip()
]

# Filtrar épocas que contienen evaluación COCO
# Si se reanudó el entrenamiento, una misma época puede aparecer repetida:
# nos quedamos con la última aparición de cada una (esto era lo que ensuciaba la gráfica)
by_epoch = {}
for r in rows:
    if "test_coco_eval_bbox" in r:
        by_epoch[r["epoch"]] = r

epochs = sorted(by_epoch)
valid_rows = [by_epoch[e] for e in epochs]

# Orden de las métricas COCO en test_coco_eval_bbox
METRIC_NAMES = [
    "AP (0.50:0.95)", "AP50", "AP75",
    "AP small", "AP medium", "AP large",
    "AR@1", "AR@10", "AR@100",
    "AR small", "AR medium", "AR large",
]

ap   = [r["test_coco_eval_bbox"][0] for r in valid_rows]
ap50 = [r["test_coco_eval_bbox"][1] for r in valid_rows]
ap75 = [r["test_coco_eval_bbox"][2] for r in valid_rows]
loss = [r.get("train_loss") for r in valid_rows]

best = max(range(len(ap)), key=lambda i: ap[i])

# ---------- Gráfica ----------
has_loss = all(l is not None for l in loss)
fig, axes = plt.subplots(
    1, 2 if has_loss else 1,
    figsize=(15 if has_loss else 9, 5),
    squeeze=False
)
ax = axes[0][0]

ax.plot(epochs, ap,   marker="o", markersize=3, linewidth=1.5, label="AP (0.50:0.95)")
ax.plot(epochs, ap50, marker="s", markersize=3, linewidth=1.5, label="AP50")
ax.plot(epochs, ap75, marker="^", markersize=3, linewidth=1.5, label="AP75")

ax.scatter([epochs[best]], [ap[best]], color="red", zorder=5, s=70,
           label=f"Mejor AP: {ap[best]:.3f} (época {epochs[best]})")
ax.axvline(epochs[best], color="red", linestyle="--", alpha=0.4)

ax.set_ylim(0, 1.0)
ax.set_xlabel("Época")
ax.set_ylabel("mAP")
ax.set_title("Evaluación RT-DETRv2 - Matrículas")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)

if has_loss:
    ax2 = axes[0][1]
    ax2.plot(epochs, loss, color="tab:orange", marker="o", markersize=3, linewidth=1.5)
    ax2.set_xlabel("Época")
    ax2.set_ylabel("Loss")
    ax2.set_title("Pérdida de entrenamiento")
    ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "curvas_ap_loss.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------- Resultados ----------
def print_metrics(title, row, epoch):
    stats = row["test_coco_eval_bbox"]
    print(f"\n{'=' * 44}")
    print(f" {title} (época {epoch})")
    print(f"{'=' * 44}")
    for name, value in zip(METRIC_NAMES, stats):
        v = "  n/a" if value < 0 else f"{value * 100:6.2f}%"
        print(f" {name:<16} {v}")

print(f"Épocas evaluadas: {len(epochs)}  (de la {epochs[0]} a la {epochs[-1]})")
print_metrics("MEJOR ÉPOCA", valid_rows[best], epochs[best])
print_metrics("ÚLTIMA ÉPOCA", valid_rows[-1], epochs[-1])

**GRÁFICAS ADICIONALES DEL PROGRESO DEL ENTRENAMIENTO**

Se generan cuatro gráficas complementarias a partir de `log.txt`, guardadas en `RT-DETR_resultado_entrenamiento/progreso_entrenamiento.png`:
- **AP por tamaño de objeto**: Evolución de AP en objetos pequeños, medianos y grandes (las matrículas pequeñas son el caso más difícil).
- **Recall medio (AR)**: Evolución de AR@1, AR@10 y AR@100.
- **Componentes de la pérdida**: Pérdida total junto con las pérdidas de clasificación (VFL), de caja (L1) y de GIoU, si están en el log.
- **Tasa de aprendizaje y mejora de AP**: Evolución del *learning rate* y de la mejora de AP respecto a la primera época.

También se guarda la tabla completa de métricas por época en `metricas_por_epoca.csv`.

In [ ]:
import csv

def serie(key):
    """Devuelve la lista de valores de 'key' por época, o None si falta en el log."""
    vals = [r.get(key) for r in valid_rows]
    return vals if all(v is not None for v in vals) else None

stats_all = [r["test_coco_eval_bbox"] for r in valid_rows]
col = lambda i: [s[i] for s in stats_all]

fig, axs = plt.subplots(2, 2, figsize=(15, 10))

# 1) AP por tamaño de objeto
a = axs[0][0]
a.plot(epochs, col(3), marker="o", markersize=3, label="AP small")
a.plot(epochs, col(4), marker="s", markersize=3, label="AP medium")
a.plot(epochs, col(5), marker="^", markersize=3, label="AP large")
a.set_title("AP por tamaño de objeto")
a.set_xlabel("Época"); a.set_ylabel("AP"); a.set_ylim(0, 1.0)
a.legend(loc="lower right"); a.grid(alpha=0.3)

# 2) Recall medio
a = axs[0][1]
a.plot(epochs, col(6), marker="o", markersize=3, label="AR@1")
a.plot(epochs, col(7), marker="s", markersize=3, label="AR@10")
a.plot(epochs, col(8), marker="^", markersize=3, label="AR@100")
a.set_title("Recall medio (AR)")
a.set_xlabel("Época"); a.set_ylabel("AR"); a.set_ylim(0, 1.0)
a.legend(loc="lower right"); a.grid(alpha=0.3)

# 3) Componentes de la pérdida
a = axs[1][0]
for key, name in [("train_loss", "Loss total"),
                  ("train_loss_vfl", "VFL (clasificación)"),
                  ("train_loss_bbox", "L1 (caja)"),
                  ("train_loss_giou", "GIoU")]:
    s = serie(key)
    if s is not None:
        a.plot(epochs, s, marker="o", markersize=3, label=name)
a.set_yscale("log")
a.set_title("Componentes de la pérdida (escala log)")
a.set_xlabel("Época"); a.set_ylabel("Loss")
a.legend(); a.grid(alpha=0.3, which="both")

# 4) Learning rate y mejora de AP respecto a la primera época
a = axs[1][1]
mejora = [(v - ap[0]) * 100 for v in ap]
a.bar(epochs, mejora, color="tab:green", alpha=0.6, label="Mejora de AP vs. 1ª época (pp)")
a.axhline(0, color="black", linewidth=0.8)
a.set_xlabel("Época"); a.set_ylabel("Puntos porcentuales")
a.set_title("Learning rate y mejora de AP")
a.grid(alpha=0.3)
lr = serie("train_lr")
if lr is not None:
    a_lr = a.twinx()
    a_lr.plot(epochs, lr, color="tab:red", marker="o", markersize=3, label="Learning rate")
    a_lr.set_ylabel("Learning rate")
    h1, l1 = a.get_legend_handles_labels()
    h2, l2 = a_lr.get_legend_handles_labels()
    a.legend(h1 + h2, l1 + l2, loc="lower right")
else:
    a.legend(loc="lower right")

plt.suptitle("Progreso del entrenamiento RT-DETRv2 - Matrículas", fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "progreso_entrenamiento.png", dpi=150, bbox_inches="tight")
plt.show()

# ---------- Tabla de métricas por época ----------
with open(RESULTS_DIR / "metricas_por_epoca.csv", "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f, delimiter=";")
    w.writerow(["epoca", "train_loss"] + METRIC_NAMES)
    for e, r in zip(epochs, valid_rows):
        w.writerow([e, r.get("train_loss", "")] + list(r["test_coco_eval_bbox"]))

print("Gráficas y tabla guardadas en:", RESULTS_DIR)

**VALIDACIÓN DEL MEJOR MODELO**

La función `run_validation` evalúa unos pesos concretos sobre el conjunto de validación ejecutando `tools/train.py` con `--test-only`. Se evalúa `best.pth`, que corresponde a la época con mejor AP.

In [ ]:
def run_validation(weights):

    cmd = [
        sys.executable,
        "tools/train.py",
        "-c", str(CONFIG),
        "-r", str(weights),
        "--test-only"
    ]

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0"
    env["PYTHONUNBUFFERED"] = "1"

    process = subprocess.Popen(
        cmd,
        cwd=str(RTDETR_PATH),
        env=env,
        text=True,
        bufsize=1,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )

    for line in process.stdout:
        print(line, end="")

    process.wait()

    if process.returncode != 0:
        raise RuntimeError("Error durante la validación")

    print("\nValidación completada.")


# Evaluar best.pth
run_validation(OUTPUT_DIR / "best.pth")

**PRUEBA DE INFERENCIA: LOTE DE IMÁGENES DE VALIDACIÓN**

Como comprobación final se carga `best.pth` (pesos EMA) con la misma configuración del entrenamiento y se ejecuta el detector sobre un lote de imágenes del conjunto de validación, con un umbral de confianza de 0.3 y la misma resolución que en el entrenamiento (640x640). En cada imagen se dibujan:
- **Cajas reales (verde)**: Anotaciones del conjunto de validación.
- **Cajas predichas (rojo)**: Detecciones del modelo junto con su confianza.

El lote se guarda en `RT-DETR_resultado_entrenamiento/lote_validacion.png`.

In [ ]:
import random
import numpy as np
import cv2
from PIL import Image

if RTDETR_PATH not in sys.path:
    sys.path.insert(0, RTDETR_PATH)

from src.core import YAMLConfig

# Parámetros del lote de imágenes
N_IMAGENES = 12      # imágenes del lote
CONF_MIN = 0.3       # confianza mínima de las detecciones mostradas
COLUMNAS = 4

# Rutas del dataset (relativas a rtdetrv2_pytorch, igual que en el .yml)
DATASET_DIR = (Path(RTDETR_PATH) / ".." / ".." / "plates" / "coco-dataset").resolve()
IMG_DIR = DATASET_DIR / "images"
VAL_JSON = DATASET_DIR / "annotations" / "instances_val.json"

# ---------- Carga del mejor modelo ----------
cfg = YAMLConfig(str(CONFIG), resume=str(OUTPUT_DIR / "best.pth"))

checkpoint = torch.load(OUTPUT_DIR / "best.pth", map_location="cpu")
state = checkpoint["ema"]["module"] if "ema" in checkpoint else checkpoint["model"]
cfg.model.load_state_dict(state)

class ModeloDeploy(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.model = cfg.model.deploy()
        self.postprocessor = cfg.postprocessor.deploy()

    def forward(self, images, orig_target_sizes):
        outputs = self.model(images)
        return self.postprocessor(outputs, orig_target_sizes)

detector = ModeloDeploy().to(DEVICE).eval()

# ---------- Selección del lote de imágenes ----------
with open(VAL_JSON, encoding="utf-8") as f:
    coco = json.load(f)

anotaciones = {}
for ann in coco["annotations"]:
    anotaciones.setdefault(ann["image_id"], []).append(ann["bbox"])   # bbox COCO: x, y, w, h

random.seed(0)
imagenes = random.sample(coco["images"], min(N_IMAGENES, len(coco["images"])))

# ---------- Inferencia y dibujado ----------
resultados = []
for info in imagenes:
    bgr = cv2.imread(str(IMG_DIR / info["file_name"]))
    if bgr is None:
        continue
    h, w = bgr.shape[:2]
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    # mismo preprocesado que en validación: resize a 640x640 y normalización 0-1
    im = Image.fromarray(rgb).resize((640, 640), Image.BILINEAR)
    x = torch.from_numpy(np.array(im)).permute(2, 0, 1).float().div(255)[None].to(DEVICE)
    size = torch.tensor([[w, h]], device=DEVICE)

    with torch.no_grad():
        labels, boxes, scores = detector(x, size)
    boxes, scores = boxes[0].cpu().numpy(), scores[0].cpu().numpy()
    keep = scores >= CONF_MIN
    boxes, scores = boxes[keep], scores[keep]

    # cajas reales en verde
    for bx, by, bw, bh in anotaciones.get(info["id"], []):
        cv2.rectangle(rgb, (int(bx), int(by)), (int(bx + bw), int(by + bh)), (0, 200, 0), max(2, w // 400))

    # cajas predichas en rojo
    for (x1, y1, x2, y2), s in zip(boxes, scores):
        cv2.rectangle(rgb, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0, 0), max(2, w // 400))
        cv2.putText(rgb, f"{s:.2f}", (int(x1), max(15, int(y1) - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, max(0.5, w / 1600), (255, 0, 0), max(1, w // 800))

    resultados.append((rgb, info["file_name"], len(boxes), len(anotaciones.get(info["id"], []))))

# ---------- Rejilla de resultados ----------
filas = int(np.ceil(len(resultados) / COLUMNAS))
fig, axs = plt.subplots(filas, COLUMNAS, figsize=(5 * COLUMNAS, 4 * filas), squeeze=False)

for ax in axs.ravel():
    ax.axis("off")
for ax, (img, nombre, n_pred, n_real) in zip(axs.ravel(), resultados):
    ax.imshow(img)
    ax.set_title(f"{nombre}\nreales: {n_real} | detectadas: {n_pred}", fontsize=9)

plt.suptitle("Lote de validación RT-DETRv2 (verde: real, rojo: predicción)", fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "lote_validacion.png", dpi=150, bbox_inches="tight")
plt.show()

print("Lote de imágenes guardado en:", RESULTS_DIR / "lote_validacion.png")